Phase 1 → Project setup + document collection

In [8]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Drive mounted")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Drive mounted


Set Paths & Verify Documents



In [9]:
from pathlib import Path

DRIVE_BASE = Path("/content/drive/MyDrive/healthcare_rag")

DOCUMENTS_DIR   = DRIVE_BASE / "documents"
DATA_DIR        = DRIVE_BASE / "data"
EMBEDDINGS_DIR  = DRIVE_BASE / "embeddings"
VECTORSTORE_DIR = DRIVE_BASE / "vectorstore"
OUTPUTS_DIR     = DRIVE_BASE / "outputs"

# Create output folders in Drive if they don't exist
for folder in [DATA_DIR, EMBEDDINGS_DIR, VECTORSTORE_DIR, OUTPUTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Verify documents
pdfs = sorted(DOCUMENTS_DIR.glob("*.pdf"))
print(f"Documents found: {len(pdfs)}")
for f in pdfs:
    print(f"  → {f.name}")

Documents found: 15
  → 01_diabetes_who.pdf
  → 02_hypertension_cdc.pdf
  → 03_asthma_who.pdf
  → 04_conjunctivitis_cdc.pdf
  → 05_jaundice_hepatitis_who.pdf
  → 06_essential_medicines_who_2023.pdf
  → 07_essential_medicines_who_2019.pdf
  → 08_hepatitis_a_symptoms_who.pdf
  → 09_conjunctivitis_diagnosis_cdc.pdf
  → 10_healthy_diet_who.pdf
  → 11_nutrition_preventive_who.pdf
  → 12_diabetes_management_procedures.pdf
  → 13_hypertension_management_cdc.pdf
  → 14_asthma_guidelines_who.pdf
  → 15_hepatitis_screening_guidelines_who.pdf


 Install Libraries

In [10]:
%pip install -q pymupdf langchain sentence-transformers chromadb faiss-cpu groq tqdm evaluate rouge-score nltk
print("✓ All libraries installed")

✓ All libraries installed


Define Config (in memory, no file needed)

In [11]:
# All config lives here in the notebook as plain variables
CHUNK_SIZES      = [256, 512, 1024]
CHUNK_OVERLAPS   = [50, 100, 150]
EMBEDDING_MODELS = ["all-MiniLM-L6-v2", "BAAI/bge-small-en-v1.5"]
VECTOR_DBS       = ["chromadb", "faiss"]
TOP_K            = 5
LLM_MODEL        = "llama-3.1-8b-instant"
LLM_TEMPERATURE  = 0.2
LLM_MAX_TOKENS   = 1024

print("✓ Config loaded")

✓ Config loaded


 Document Loader Class

In [12]:
import fitz  # PyMuPDF
from dataclasses import dataclass, asdict
from typing import List

@dataclass
class Document:
    filename: str
    file_type: str
    page_number: int
    raw_text: str
    character_count: int

class DocumentLoader:

    def __init__(self, documents_dir: Path):
        self.documents_dir = documents_dir
        self.loaded_documents: List[Document] = []

    def load_pdf(self, filepath: Path) -> List[Document]:
        docs = []
        try:
            pdf = fitz.open(str(filepath))
            for page_num in range(len(pdf)):
                text = pdf[page_num].get_text().strip()
                if text:
                    docs.append(Document(
                        filename=filepath.name,
                        file_type="pdf",
                        page_number=page_num + 1,
                        raw_text=text,
                        character_count=len(text)
                    ))
            pdf.close()
        except Exception as e:
            print(f"  ✗ Error: {filepath.name} → {e}")
        return docs

    def load_all(self) -> List[Document]:
        files = sorted(self.documents_dir.glob("*.pdf"))
        if not files:
            print("⚠ No PDFs found in documents/ folder")
            return []

        print(f"Found {len(files)} files\n")
        self.loaded_documents = []

        for i, filepath in enumerate(files, 1):
            docs = self.load_pdf(filepath)
            self.loaded_documents.extend(docs)
            chars = sum(d.character_count for d in docs)
            print(f"[{i:02}/{len(files)}] {filepath.name:<50} pages: {len(docs)}   chars: {chars:,}")

        return self.loaded_documents

    def get_statistics(self):
        total_chars = sum(d.character_count for d in self.loaded_documents)
        unique_files = list(dict.fromkeys(d.filename for d in self.loaded_documents))

        print("\n" + "="*60)
        print("DOCUMENT STATISTICS")
        print("="*60)
        print(f"Total Files    : {len(unique_files)}")
        print(f"Total Pages    : {len(self.loaded_documents)}")
        print(f"Total Chars    : {total_chars:,}")
        print("-"*60)
        for fname in unique_files:
            file_docs = [d for d in self.loaded_documents if d.filename == fname]
            print(f"  {fname:<48} {len(file_docs):>2} pages  {sum(d.character_count for d in file_docs):>8,} chars")
        print("="*60)

print("✓ DocumentLoader class defined")

✓ DocumentLoader class defined


Run Loader & Save to Drive

In [13]:
import json

loader = DocumentLoader(DOCUMENTS_DIR)
documents = loader.load_all()
loader.get_statistics()

# Save to Drive
output_path = DATA_DIR / "loaded_documents.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump([asdict(d) for d in documents], f, indent=2, ensure_ascii=False)

print(f"\n✓ Saved to Drive: {output_path}")
print(f"  Total entries: {len(documents)}")

Found 15 files

[01/15] 01_diabetes_who.pdf                                pages: 2   chars: 6,105
[02/15] 02_hypertension_cdc.pdf                            pages: 8   chars: 17,537
[03/15] 03_asthma_who.pdf                                  pages: 10   chars: 43,339
[04/15] 04_conjunctivitis_cdc.pdf                          pages: 1   chars: 1,588
[05/15] 05_jaundice_hepatitis_who.pdf                      pages: 45   chars: 26,471
[06/15] 06_essential_medicines_who_2023.pdf                pages: 70   chars: 163,330
[07/15] 07_essential_medicines_who_2019.pdf                pages: 64   chars: 110,263
[08/15] 08_hepatitis_a_symptoms_who.pdf                    pages: 5   chars: 9,640
[09/15] 09_conjunctivitis_diagnosis_cdc.pdf                pages: 4   chars: 6,288
[10/15] 10_healthy_diet_who.pdf                            pages: 6   chars: 19,058
[11/15] 11_nutrition_preventive_who.pdf                    pages: 5   chars: 10,744
[12/15] 12_diabetes_management_procedures.pdf             

Verify

In [14]:
import json

json_path = DATA_DIR / "loaded_documents.json"
with open(json_path) as f:
    data = json.load(f)

empty = [d for d in data if not d["raw_text"].strip()]

print("="*60)
print("PHASE 1 VERIFICATION")
print("="*60)
print(f"✓ JSON entries      : {len(data)}")
print(f"{'✓' if not empty else '⚠'} Empty text entries : {len(empty)}")

if empty:
    print("  Files with empty text — replace these PDFs:")
    for e in empty:
        print(f"    → {e['filename']} page {e['page_number']}")

print("\nPhase 1 Complete ✓" if not empty else "\nFix empty PDFs then rerun")

PHASE 1 VERIFICATION
✓ JSON entries      : 377
✓ Empty text entries : 0

Phase 1 Complete ✓


-----Phase 2 — Text Chunking & Preprocessing----

Load Documents from Drive


In [15]:
import json
from pathlib import Path

DRIVE_BASE    = Path("/content/drive/MyDrive/healthcare_rag")
DATA_DIR      = DRIVE_BASE / "data"

json_path = DATA_DIR / "loaded_documents.json"
with open(json_path, "r", encoding="utf-8") as f:
    raw_documents = json.load(f)

print(f"✓ Loaded {len(raw_documents)} document pages from Drive")
print(f"  Sample entry keys: {list(raw_documents[0].keys())}")
print(f"\n  First entry preview:")
print(f"  File     : {raw_documents[0]['filename']}")
print(f"  Page     : {raw_documents[0]['page_number']}")
print(f"  Chars    : {raw_documents[0]['character_count']}")
print(f"  Text[:100]: {raw_documents[0]['raw_text'][:100]}...")

✓ Loaded 377 document pages from Drive
  Sample entry keys: ['filename', 'file_type', 'page_number', 'raw_text', 'character_count']

  First entry preview:
  File     : 01_diabetes_who.pdf
  Page     : 1
  Chars    : 3019
  Text[:100]: Quick facts
Globally, an estimated 346 million people have diabetes. Three out of four people with d...


Text Cleaning Function

In [16]:
import re

def clean_text(text: str, lowercase: bool = False) -> str:
    """
    Clean raw extracted PDF text for chunking.
    - Removes excessive whitespace and newlines
    - Removes non-medical special characters
    - Removes likely header/footer lines (very short repeated lines)
    - Optionally lowercases text
    """
    # Remove hyphenated line breaks (PDF artifact: "diabe-\ntes" → "diabetes")
    text = re.sub(r'-\n', '', text)

    # Replace newlines and tabs with space
    text = re.sub(r'[\n\t\r]+', ' ', text)

    # Remove characters that are not medically useful
    # Keep: letters, numbers, spaces, . , - ( ) / % : ; ' "
    text = re.sub(r'[^\w\s\.\,\-\(\)\/\%\:\;\'\"\+\=]', ' ', text)

    # Collapse multiple spaces into one
    text = re.sub(r' {2,}', ' ', text)

    # Strip leading/trailing whitespace
    text = text.strip()

    if lowercase:
        text = text.lower()

    return text


def remove_short_lines(text: str, min_length: int = 40) -> str:
    """
    Remove lines shorter than min_length characters.
    These are usually page numbers, headers, footers.
    """
    lines = text.split('.')
    filtered = [line for line in lines if len(line.strip()) >= min_length]
    return '. '.join(filtered)


# Show before and after on one real page
sample = raw_documents[10]
original = sample['raw_text']
cleaned  = clean_text(original)
cleaned  = remove_short_lines(cleaned)

print("=" * 60)
print("TEXT CLEANING — BEFORE vs AFTER")
print("=" * 60)
print(f"\nFile: {sample['filename']}  |  Page: {sample['page_number']}")
print(f"\n--- BEFORE ({len(original)} chars) ---")
print(original[:400])
print(f"\n--- AFTER  ({len(cleaned)} chars) ---")
print(cleaned[:400])
print(f"\nCharacters removed: {len(original) - len(cleaned):,}")
print("=" * 60)

TEXT CLEANING — BEFORE vs AFTER

File: 03_asthma_who.pdf  |  Page: 1

--- BEFORE (4017 chars) ---
▪ Babies born to mothers who smoke have smaller lungs and an increased risk of developing asthma 
during childhood. Pregnant women should receive targeted support to quit tobacco use. E-cigarettes, 
heated tobacco products and other nicotine-delivery devices likely also carry risks.
▪ Children exposed to second-hand tobacco smoke have an increased risk of developing asthma. 
▪ Smoking during adole

--- AFTER  (3961 chars) ---
Babies born to mothers who smoke have smaller lungs and an increased risk of developing asthma during childhood.  Pregnant women should receive targeted support to quit tobacco use.  E-cigarettes, heated tobacco products and other nicotine-delivery devices likely also carry risks.  Children exposed to second-hand tobacco smoke have an increased risk of developing asthma.  Smoking during adolescenc

Characters removed: 56


Apply Cleaning to All Documents

In [17]:
from tqdm import tqdm

cleaned_documents = []

for doc in tqdm(raw_documents, desc="Cleaning documents"):
    cleaned_text = clean_text(doc['raw_text'])
    cleaned_text = remove_short_lines(cleaned_text)

    cleaned_documents.append({
        "filename"        : doc['filename'],
        "file_type"       : doc['file_type'],
        "page_number"     : doc['page_number'],
        "raw_text"        : doc['raw_text'],
        "cleaned_text"    : cleaned_text,
        "original_chars"  : doc['character_count'],
        "cleaned_chars"   : len(cleaned_text)
    })

total_original = sum(d['original_chars'] for d in cleaned_documents)
total_cleaned  = sum(d['cleaned_chars']  for d in cleaned_documents)

print(f"\n✓ Cleaned {len(cleaned_documents)} pages")
print(f"  Original total chars : {total_original:,}")
print(f"  Cleaned  total chars : {total_cleaned:,}")
print(f"  Chars removed        : {total_original - total_cleaned:,}")
print(f"  Reduction            : {((total_original - total_cleaned)/total_original)*100:.1f}%")

Cleaning documents: 100%|██████████| 377/377 [00:00<00:00, 2671.09it/s]


✓ Cleaned 377 pages
  Original total chars : 709,372
  Cleaned  total chars : 594,509
  Chars removed        : 114,863
  Reduction            : 16.2%


 Implement 3 Chunking Strategies

In [19]:
%pip install -q langchain langchain-text-splitters

# Updated import path for newer LangChain versions
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_documents(documents: list, chunk_size: int, chunk_overlap: int) -> list:
    """
    Split all cleaned document pages into chunks using
    RecursiveCharacterTextSplitter with given size and overlap.
    Each chunk carries full metadata.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size      = chunk_size,
        chunk_overlap   = chunk_overlap,
        length_function = len,
        separators      = [". ", "! ", "? ", "\n", " ", ""]
    )

    all_chunks = []
    chunk_index = 0

    for doc in documents:
        text = doc['cleaned_text']
        if not text.strip():
            continue

        splits = splitter.split_text(text)

        for split in splits:
            if split.strip():
                all_chunks.append({
                    "chunk_id"          : chunk_index,
                    "source_filename"   : doc['filename'],
                    "page_number"       : doc['page_number'],
                    "chunk_size_config" : chunk_size,
                    "chunk_overlap"     : chunk_overlap,
                    "text"              : split.strip(),
                    "character_count"   : len(split.strip())
                })
                chunk_index += 1

    return all_chunks


print("Chunking with Strategy A — size=256,  overlap=50  ...")
chunks_256  = chunk_documents(cleaned_documents, chunk_size=256,  chunk_overlap=50)
print(f"  ✓ {len(chunks_256):,} chunks produced")

print("Chunking with Strategy B — size=512,  overlap=100 ...")
chunks_512  = chunk_documents(cleaned_documents, chunk_size=512,  chunk_overlap=100)
print(f"  ✓ {len(chunks_512):,} chunks produced")

print("Chunking with Strategy C — size=1024, overlap=150 ...")
chunks_1024 = chunk_documents(cleaned_documents, chunk_size=1024, chunk_overlap=150)
print(f"  ✓ {len(chunks_1024):,} chunks produced")

Chunking with Strategy A — size=256,  overlap=50  ...
  ✓ 3,303 chunks produced
Chunking with Strategy B — size=512,  overlap=100 ...
  ✓ 1,588 chunks produced
Chunking with Strategy C — size=1024, overlap=150 ...
  ✓ 828 chunks produced


Chunking Statistics & Comparison Table

In [20]:
import numpy as np

def compute_stats(chunks: list, label: str) -> dict:
    """Compute descriptive statistics for a chunk set."""
    sizes = [c['character_count'] for c in chunks]
    files = list(set(c['source_filename'] for c in chunks))

    return {
        "Strategy"              : label,
        "Total Chunks"          : len(chunks),
        "Avg Chars/Chunk"       : int(np.mean(sizes)),
        "Min Chars"             : int(np.min(sizes)),
        "Max Chars"             : int(np.max(sizes)),
        "Std Dev"               : int(np.std(sizes)),
        "Avg Chunks/Document"   : round(len(chunks) / len(files), 1)
    }

stats_256  = compute_stats(chunks_256,  "A — size=256,  overlap=50")
stats_512  = compute_stats(chunks_512,  "B — size=512,  overlap=100")
stats_1024 = compute_stats(chunks_1024, "C — size=1024, overlap=150")

# Print comparison table
header = f"{'Metric':<28} {'Strategy A':>15} {'Strategy B':>15} {'Strategy C':>15}"
divider = "-" * 75

print("\n" + "=" * 75)
print("CHUNKING STRATEGY COMPARISON TABLE")
print("(Copy this into your assignment report)")
print("=" * 75)
print(header)
print(divider)

metrics = ["Total Chunks", "Avg Chars/Chunk", "Min Chars",
           "Max Chars", "Std Dev", "Avg Chunks/Document"]

for metric in metrics:
    a = str(stats_256[metric])
    b = str(stats_512[metric])
    c = str(stats_1024[metric])
    print(f"{metric:<28} {a:>15} {b:>15} {c:>15}")

print("=" * 75)
print("\nKey Insight:")
print(f"  Strategy A produces {len(chunks_256):,} chunks — finest granularity")
print(f"  Strategy B produces {len(chunks_512):,} chunks — balanced")
print(f"  Strategy C produces {len(chunks_1024):,} chunks — most context per chunk")


CHUNKING STRATEGY COMPARISON TABLE
(Copy this into your assignment report)
Metric                            Strategy A      Strategy B      Strategy C
---------------------------------------------------------------------------
Total Chunks                            3303            1588             828
Avg Chars/Chunk                          188             397             756
Min Chars                                 43              50              59
Max Chars                                256             512            1024
Std Dev                                   56             104             263
Avg Chunks/Document                    220.2           105.9            55.2

Key Insight:
  Strategy A produces 3,303 chunks — finest granularity
  Strategy B produces 1,588 chunks — balanced
  Strategy C produces 828 chunks — most context per chunk


 Save All Chunk Sets to Drive

In [21]:
def save_chunks(chunks: list, filename: str, data_dir: Path):
    """Save chunk list as JSON to Drive."""
    output_path = data_dir / filename
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(chunks, f, indent=2, ensure_ascii=False)
    size_mb = output_path.stat().st_size / (1024 * 1024)
    print(f"  ✓ {filename:<25} {len(chunks):>6,} chunks   {size_mb:.2f} MB")

print("Saving chunk files to Drive...\n")
save_chunks(chunks_256,  "chunks_256.json",  DATA_DIR)
save_chunks(chunks_512,  "chunks_512.json",  DATA_DIR)
save_chunks(chunks_1024, "chunks_1024.json", DATA_DIR)

# Also save cleaned documents for reference
cleaned_path = DATA_DIR / "cleaned_documents.json"
with open(cleaned_path, "w", encoding="utf-8") as f:
    json.dump(cleaned_documents, f, indent=2, ensure_ascii=False)
print(f"  ✓ {'cleaned_documents.json':<25} {len(cleaned_documents):>6,} pages")

print(f"\n✓ All files saved to {DATA_DIR}")

Saving chunk files to Drive...

  ✓ chunks_256.json            3,303 chunks   1.26 MB
  ✓ chunks_512.json            1,588 chunks   0.92 MB
  ✓ chunks_1024.json             828 chunks   0.77 MB
  ✓ cleaned_documents.json       377 pages

✓ All files saved to /content/drive/MyDrive/healthcare_rag/data


Verification

In [22]:
print("=" * 60)
print("PHASE 2 VERIFICATION")
print("=" * 60)

files_to_check = {
    "chunks_256.json"        : len(chunks_256),
    "chunks_512.json"        : len(chunks_512),
    "chunks_1024.json"       : len(chunks_1024),
    "cleaned_documents.json" : len(cleaned_documents)
}

all_good = True
for filename, expected_count in files_to_check.items():
    path = DATA_DIR / filename
    if path.exists():
        with open(path) as f:
            loaded = json.load(f)
        match = "✓" if len(loaded) == expected_count else "✗"
        if match == "✗":
            all_good = False
        print(f"  {match} {filename:<30} {len(loaded):,} entries")
    else:
        print(f"  ✗ {filename} — NOT FOUND")
        all_good = False

print("\n--- Sample Chunk from Each Strategy ---\n")
for label, chunks in [("A-256", chunks_256), ("B-512", chunks_512), ("C-1024", chunks_1024)]:
    sample = chunks[50]
    print(f"Strategy {label}:")
    print(f"  Source   : {sample['source_filename']}")
    print(f"  Page     : {sample['page_number']}")
    print(f"  Chars    : {sample['character_count']}")
    print(f"  Text     : {sample['text'][:120]}...")
    print()

print("=" * 60)
if all_good:
    print("Phase 2 Complete ✓ — Ready for Phase 3")
else:
    print("⚠ Some files missing — rerun saving cells")
print("=" * 60)

PHASE 2 VERIFICATION
  ✓ chunks_256.json                3,303 entries
  ✓ chunks_512.json                1,588 entries
  ✓ chunks_1024.json               828 entries
  ✓ cleaned_documents.json         377 entries

--- Sample Chunk from Each Strategy ---

Strategy A-256:
  Source   : 02_hypertension_cdc.pdf
  Page     : 2
  Chars    : 247
  Text     : .  Among women, the age-adjusted prevalence of hypertension was higher among non-Hispanic black (56.  Both non-Hispanic ...

Strategy B-512:
  Source   : 02_hypertension_cdc.pdf
  Page     : 8
  Chars    : 491
  Text     : 364    April 2020 Keywords: high blood pressure National Health and Nutrition Examination Survey (NHANES) Suggested cita...

Strategy C-1024:
  Source   : 03_asthma_who.pdf
  Page     : 5
  Chars    : 814
  Text     : .  It helps users identify triggers, set targets, manage craving and stay focused on quitting tobacco.  Emerging concern...

Phase 2 Complete ✓ — Ready for Phase 3


In [1]:
from google.colab import drive
drive.mount('/content/drive')

%pip install -q pymupdf
%pip install -q langchain langchain-text-splitters
%pip install -q sentence-transformers
%pip install -q chromadb
%pip install -q faiss-cpu
%pip install -q tqdm
%pip install -q numpy

print("✓ Drive mounted and libraries installed")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 81.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 97.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.4 MB/s eta 0:00:00
ERROR: pip's dependency r

In [2]:
import json
from pathlib import Path

DRIVE_BASE      = Path("/content/drive/MyDrive/healthcare_rag")
DATA_DIR        = DRIVE_BASE / "data"
EMBEDDINGS_DIR  = DRIVE_BASE / "embeddings"
VECTORSTORE_DIR = DRIVE_BASE / "vectorstore"

# Reload all three chunk sets from Drive
with open(DATA_DIR / "chunks_256.json",  "r") as f:
    chunks_256  = json.load(f)
with open(DATA_DIR / "chunks_512.json",  "r") as f:
    chunks_512  = json.load(f)
with open(DATA_DIR / "chunks_1024.json", "r") as f:
    chunks_1024 = json.load(f)

print(f"✓ chunks_256  reloaded: {len(chunks_256):,}")
print(f"✓ chunks_512  reloaded: {len(chunks_512):,}")
print(f"✓ chunks_1024 reloaded: {len(chunks_1024):,}")
print(f"\nTotal chunks across all strategies: {len(chunks_256)+len(chunks_512)+len(chunks_1024):,}")

✓ chunks_256  reloaded: 3,303
✓ chunks_512  reloaded: 1,588
✓ chunks_1024 reloaded: 828

Total chunks across all strategies: 5,719


-------Embedding Generation & Vector Database--------

In [3]:
import torch

print("="*50)
print("GPU STATUS CHECK")
print("="*50)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✓ GPU Available  : {gpu_name}")
    print(f"✓ GPU Memory     : {gpu_mem:.1f} GB")
    DEVICE = "cuda"
else:
    print("⚠ No GPU found — running on CPU (will be slower)")
    print("  Go to Runtime → Change runtime type → T4 GPU")
    DEVICE = "cpu"

print(f"\nUsing device: {DEVICE}")

GPU STATUS CHECK
✓ GPU Available  : Tesla T4
✓ GPU Memory     : 15.6 GB

Using device: cuda


 Load Both Embedding Models

In [4]:
from sentence_transformers import SentenceTransformer
import time

EMBEDDING_MODELS = {
    "minilm"  : "all-MiniLM-L6-v2",
    "bge"     : "BAAI/bge-small-en-v1.5"
}

loaded_models = {}

print("Loading embedding models...\n")
for model_key, model_name in EMBEDDING_MODELS.items():
    print(f"Loading {model_name}...")
    start = time.time()
    model = SentenceTransformer(model_name, device=DEVICE)
    elapsed = time.time() - start

    # Get embedding dimension by encoding a test sentence
    test_vector = model.encode("test sentence")
    dim = len(test_vector)

    loaded_models[model_key] = model
    print(f"  ✓ Loaded in {elapsed:.1f}s")
    print(f"  ✓ Embedding dimensions : {dim}")
    print(f"  ✓ Device               : {DEVICE}\n")

print("✓ Both embedding models ready")

Loading embedding models...

Loading all-MiniLM-L6-v2...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  ✓ Loaded in 5.7s
  ✓ Embedding dimensions : 384
  ✓ Device               : cuda

✓ Both embedding models ready


Generate Embeddings for All Chunk Sets

In [5]:
import numpy as np
from tqdm import tqdm

def generate_embeddings(chunks: list, model, model_name: str, batch_size: int = 64) -> np.ndarray:
    """
    Generate embeddings for a list of chunks in batches.
    Batching means we process 64 chunks at a time instead of
    one by one — much faster especially on GPU.
    """
    texts = [chunk['text'] for chunk in chunks]
    all_embeddings = []

    print(f"  Generating embeddings with {model_name}")
    print(f"  Total texts : {len(texts):,}")
    print(f"  Batch size  : {batch_size}")

    for i in tqdm(range(0, len(texts), batch_size), desc=f"  Embedding"):
        batch = texts[i : i + batch_size]
        batch_embeddings = model.encode(
            batch,
            convert_to_numpy = True,
            show_progress_bar = False,
            normalize_embeddings = True   # normalize for cosine similarity
        )
        all_embeddings.append(batch_embeddings)

    return np.vstack(all_embeddings)


# Storage for all embeddings
# Structure: embeddings[model_key][chunk_size] = numpy array
all_embeddings = {}

for model_key, model in loaded_models.items():
    model_name = EMBEDDING_MODELS[model_key]
    all_embeddings[model_key] = {}
    print(f"\n{'='*55}")
    print(f"Model: {model_name}")
    print(f"{'='*55}")

    for chunk_label, chunks in [("256", chunks_256), ("512", chunks_512), ("1024", chunks_1024)]:
        print(f"\nChunk size {chunk_label}:")
        start = time.time()
        embeddings = generate_embeddings(chunks, model, model_name)
        elapsed = time.time() - start
        all_embeddings[model_key][chunk_label] = embeddings
        print(f"  ✓ Shape   : {embeddings.shape}  (chunks × dimensions)")
        print(f"  ✓ Time    : {elapsed:.1f}s")
        print(f"  ✓ Memory  : {embeddings.nbytes / 1e6:.1f} MB")

print("\n✓ All embeddings generated")


Model: all-MiniLM-L6-v2

Chunk size 256:
  Generating embeddings with all-MiniLM-L6-v2
  Total texts : 3,303
  Batch size  : 64


  Embedding: 100%|██████████| 52/52 [00:02<00:00, 21.45it/s]


  ✓ Shape   : (3303, 384)  (chunks × dimensions)
  ✓ Time    : 2.4s
  ✓ Memory  : 5.1 MB

Chunk size 512:
  Generating embeddings with all-MiniLM-L6-v2
  Total texts : 1,588
  Batch size  : 64


  Embedding: 100%|██████████| 25/25 [00:01<00:00, 13.24it/s]


  ✓ Shape   : (1588, 384)  (chunks × dimensions)
  ✓ Time    : 1.9s
  ✓ Memory  : 2.4 MB

Chunk size 1024:
  Generating embeddings with all-MiniLM-L6-v2
  Total texts : 828
  Batch size  : 64


...

  ✓ Shape   : (3303, 384)  (chunks × dimensions)
  ✓ Time    : 5.5s
  ✓ Memory  : 5.1 MB

Chunk size 512:
  Generating embeddings with BAAI/bge-small-en-v1.5
  Total texts : 1,588
  Batch size  : 64


  Embedding: 100%|██████████| 25/25 [00:03<00:00,  7.59it/s]


  ✓ Shape   : (1588, 384)  (chunks × dimensions)
  ✓ Time    : 3.3s
  ✓ Memory  : 2.4 MB

Chunk size 1024:
  Generating embeddings with BAAI/bge-small-en-v1.5
  Total texts : 828
  Batch size  : 64


  Embedding: 100%|██████████| 13/13 [00:03<00:00,  3.80it/s]

  ✓ Shape   : (828, 384)  (chunks × dimensions)
  ✓ Time    : 3.4s
  ✓ Memory  : 1.3 MB

✓ All embeddings generated


Save Embeddings to Drive

In [6]:
"""
We save embeddings as .npy files (numpy format).
This means we never have to regenerate them — just reload from Drive.
Regenerating would take minutes each time, loading takes seconds.
"""

print("Saving embeddings to Drive...\n")

for model_key in all_embeddings:
    for chunk_label in all_embeddings[model_key]:
        filename = f"embeddings_{model_key}_chunks{chunk_label}.npy"
        save_path = EMBEDDINGS_DIR / filename
        np.save(str(save_path), all_embeddings[model_key][chunk_label])
        size_mb = save_path.stat().st_size / 1e6
        shape   = all_embeddings[model_key][chunk_label].shape
        print(f"  ✓ {filename:<45} shape: {str(shape):<20} {size_mb:.1f} MB")

print(f"\n✓ All embeddings saved to {EMBEDDINGS_DIR}")

Saving embeddings to Drive...

  ✓ embeddings_minilm_chunks256.npy               shape: (3303, 384)          5.1 MB
  ✓ embeddings_minilm_chunks512.npy               shape: (1588, 384)          2.4 MB
  ✓ embeddings_minilm_chunks1024.npy              shape: (828, 384)           1.3 MB
  ✓ embeddings_bge_chunks256.npy                  shape: (3303, 384)          5.1 MB
  ✓ embeddings_bge_chunks512.npy                  shape: (1588, 384)          2.4 MB
  ✓ embeddings_bge_chunks1024.npy                 shape: (828, 384)           1.3 MB

✓ All embeddings saved to /content/drive/MyDrive/healthcare_rag/embeddings


Build ChromaDB Vector Stores

In [7]:
"""
ChromaDB is a vector database that:
- Persists to disk (your Drive) so it survives session restarts
- Lets you search by meaning (semantic search) not just keywords
- Stores both the vector AND the original text together
"""

import chromadb
import sys

print("Building ChromaDB vector stores...\n")
print("ChromaDB stores vectors persistently on Drive.")
print("Each collection = one combination of model + chunk size\n")

# Point ChromaDB to Drive for persistence
chroma_path = str(VECTORSTORE_DIR / "chromadb")
chroma_client = chromadb.PersistentClient(path=chroma_path)

chroma_collections = {}

chunk_map = {
    "256"  : chunks_256,
    "512"  : chunks_512,
    "1024" : chunks_1024
}

for model_key in loaded_models:
    chroma_collections[model_key] = {}
    for chunk_label, chunks in chunk_map.items():
        collection_name = f"{model_key}_chunks{chunk_label}"

        # Delete if exists so we can rebuild cleanly
        try:
            chroma_client.delete_collection(collection_name)
        except:
            pass

        collection = chroma_client.create_collection(
            name     = collection_name,
            metadata = {"hnsw:space": "cosine"}  # use cosine similarity
        )

        embeddings = all_embeddings[model_key][chunk_label]

        # ChromaDB needs: ids, embeddings, documents, metadatas
        ids       = [str(c['chunk_id'])      for c in chunks]
        documents = [c['text']               for c in chunks]
        metadatas = [{
            "source_filename"   : c['source_filename'],
            "page_number"       : c['page_number'],
            "chunk_size_config" : c['chunk_size_config'],
            "character_count"   : c['character_count']
        } for c in chunks]

        # Insert in batches of 500
        batch_size = 500
        for i in range(0, len(ids), batch_size):
            collection.add(
                ids        = ids[i:i+batch_size],
                embeddings = embeddings[i:i+batch_size].tolist(),
                documents  = documents[i:i+batch_size],
                metadatas  = metadatas[i:i+batch_size]
            )

        chroma_collections[model_key][chunk_label] = collection
        print(f"  ✓ {collection_name:<35} {collection.count():,} vectors stored")

print(f"\n✓ ChromaDB collections saved to Drive: {chroma_path}")

Building ChromaDB vector stores...

ChromaDB stores vectors persistently on Drive.
Each collection = one combination of model + chunk size

  ✓ minilm_chunks256                    3,303 vectors stored
  ✓ minilm_chunks512                    1,588 vectors stored
  ✓ minilm_chunks1024                   828 vectors stored
  ✓ bge_chunks256                       3,303 vectors stored
  ✓ bge_chunks512                       1,588 vectors stored
  ✓ bge_chunks1024                      828 vectors stored

✓ ChromaDB collections saved to Drive: /content/drive/MyDrive/healthcare_rag/vectorstore/chromadb


Build FAISS Vector Stores

In [8]:
"""
FAISS (Facebook AI Similarity Search) is a vector database that:
- Runs entirely in memory — extremely fast search
- Does not persist automatically — we save index files manually to Drive
- Best for speed comparison against ChromaDB
- Uses the same cosine similarity but with a different internal algorithm
"""

import faiss

print("Building FAISS indexes...\n")
print("FAISS runs in memory but we save index files to Drive.\n")

faiss_indexes = {}

for model_key in loaded_models:
    faiss_indexes[model_key] = {}
    for chunk_label, chunks in chunk_map.items():
        embeddings = all_embeddings[model_key][chunk_label]
        dimension  = embeddings.shape[1]

        # IndexFlatIP = Inner Product similarity (same as cosine when normalized)
        index = faiss.IndexFlatIP(dimension)
        index.add(embeddings.astype(np.float32))

        # Save to Drive
        index_filename = f"faiss_{model_key}_chunks{chunk_label}.index"
        index_path     = str(VECTORSTORE_DIR / index_filename)
        faiss.write_index(index, index_path)

        size_mb = Path(index_path).stat().st_size / 1e6
        faiss_indexes[model_key][chunk_label] = index

        print(f"  ✓ {index_filename:<45} {index.ntotal:,} vectors   {size_mb:.1f} MB")

print(f"\n✓ All FAISS indexes saved to Drive: {VECTORSTORE_DIR}")

Building FAISS indexes...

FAISS runs in memory but we save index files to Drive.

  ✓ faiss_minilm_chunks256.index                  3,303 vectors   5.1 MB
  ✓ faiss_minilm_chunks512.index                  1,588 vectors   2.4 MB
  ✓ faiss_minilm_chunks1024.index                 828 vectors   1.3 MB
  ✓ faiss_bge_chunks256.index                     3,303 vectors   5.1 MB
  ✓ faiss_bge_chunks512.index                     1,588 vectors   2.4 MB
  ✓ faiss_bge_chunks1024.index                    828 vectors   1.3 MB

✓ All FAISS indexes saved to Drive: /content/drive/MyDrive/healthcare_rag/vectorstore


Test Retrieval on Both Databases

In [9]:
"""
Quick sanity check — ask a medical question and see what
both databases retrieve. This confirms everything is working
before we build the full pipeline in Phase 4.
"""

def test_chromadb(collection, query_text: str, model, top_k: int = 3):
    """Retrieve top-k chunks from ChromaDB for a query."""
    query_vector = model.encode(query_text, normalize_embeddings=True).tolist()
    results = collection.query(
        query_embeddings = [query_vector],
        n_results        = top_k,
        include          = ["documents", "metadatas", "distances"]
    )
    return results

def test_faiss(index, chunks: list, query_text: str, model, top_k: int = 3):
    """Retrieve top-k chunks from FAISS for a query."""
    query_vector = model.encode(
        query_text,
        normalize_embeddings = True
    ).reshape(1, -1).astype(np.float32)

    scores, indices = index.search(query_vector, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "text"   : chunks[idx]['text'],
            "source" : chunks[idx]['source_filename'],
            "page"   : chunks[idx]['page_number'],
            "score"  : float(score)
        })
    return results


TEST_QUERY = "What are the symptoms of diabetes?"
MODEL_KEY  = "minilm"
CHUNK_KEY  = "512"

print("=" * 60)
print(f"TEST QUERY: '{TEST_QUERY}'")
print("=" * 60)

# ChromaDB test
print("\n--- ChromaDB Results ---")
chroma_results = test_chromadb(
    chroma_collections[MODEL_KEY][CHUNK_KEY],
    TEST_QUERY,
    loaded_models[MODEL_KEY]
)
for i, (doc, meta, dist) in enumerate(zip(
    chroma_results['documents'][0],
    chroma_results['metadatas'][0],
    chroma_results['distances'][0]
), 1):
    print(f"\n  Result {i}:")
    print(f"  Source : {meta['source_filename']}  page {meta['page_number']}")
    print(f"  Score  : {dist:.4f}")
    print(f"  Text   : {doc[:150]}...")

# FAISS test
print("\n--- FAISS Results ---")
faiss_results = test_faiss(
    faiss_indexes[MODEL_KEY][CHUNK_KEY],
    chunks_512,
    TEST_QUERY,
    loaded_models[MODEL_KEY]
)
for i, result in enumerate(faiss_results, 1):
    print(f"\n  Result {i}:")
    print(f"  Source : {result['source']}  page {result['page']}")
    print(f"  Score  : {result['score']:.4f}")
    print(f"  Text   : {result['text'][:150]}...")

print("\n✓ Both databases retrieving results correctly")

TEST QUERY: 'What are the symptoms of diabetes?'

--- ChromaDB Results ---

  Result 1:
  Source : 01_diabetes_who.pdf  page 1
  Score  : 0.2559
  Text   : .  What are the symptoms of diabetes Type 1 diabetes: Symptoms include frequent urination (polyuria), excessive thirst (polydipsia), constant hunger, ...

  Result 2:
  Source : 12_diabetes_management_procedures.pdf  page 2
  Score  : 0.2966
  Text   : .  In type 2 diabetes, the symptoms can be mild and may take many years to be noticed.  Symptoms of diabetes include: feeling very thirsty needing to ...

  Result 3:
  Source : 12_diabetes_management_procedures.pdf  page 2
  Score  : 0.4516
  Text   : .  Since 2000, mortality rates from diabetes have been increasing.  By contrast, the probability of dying from any one of the four main noncommunicabl...

--- FAISS Results ---

  Result 1:
  Source : 01_diabetes_who.pdf  page 1
  Score  : 0.7441
  Text   : .  What are the symptoms of diabetes Type 1 diabetes: Symptoms include frequent 

Verification

In [10]:
import os

print("=" * 60)
print("PHASE 3 VERIFICATION")
print("=" * 60)

print("\nEmbedding files on Drive:")
for f in sorted(EMBEDDINGS_DIR.glob("*.npy")):
    size_mb = f.stat().st_size / 1e6
    arr     = np.load(str(f))
    print(f"  ✓ {f.name:<48} shape: {str(arr.shape):<18} {size_mb:.1f} MB")

print("\nFAISS index files on Drive:")
for f in sorted(VECTORSTORE_DIR.glob("*.index")):
    size_mb = f.stat().st_size / 1e6
    print(f"  ✓ {f.name:<48} {size_mb:.1f} MB")

print("\nChromaDB folder:")
chroma_dir = VECTORSTORE_DIR / "chromadb"
if chroma_dir.exists():
    print(f"  ✓ chromadb/ folder exists at {chroma_dir}")

print("\nCollection summary:")
for model_key in chroma_collections:
    for chunk_label in chroma_collections[model_key]:
        col   = chroma_collections[model_key][chunk_label]
        fidx  = faiss_indexes[model_key][chunk_label]
        print(f"  {model_key}_chunks{chunk_label:<6}  ChromaDB: {col.count():,} vectors   FAISS: {fidx.ntotal:,} vectors")

print("\n" + "=" * 60)
print("Phase 3 Complete ✓ — Ready for Phase 4")
print("=" * 60)

PHASE 3 VERIFICATION

Embedding files on Drive:
  ✓ embeddings_bge_chunks1024.npy                    shape: (828, 384)         1.3 MB
  ✓ embeddings_bge_chunks256.npy                     shape: (3303, 384)        5.1 MB
  ✓ embeddings_bge_chunks512.npy                     shape: (1588, 384)        2.4 MB
  ✓ embeddings_minilm_chunks1024.npy                 shape: (828, 384)         1.3 MB
  ✓ embeddings_minilm_chunks256.npy                  shape: (3303, 384)        5.1 MB
  ✓ embeddings_minilm_chunks512.npy                  shape: (1588, 384)        2.4 MB

FAISS index files on Drive:
  ✓ faiss_bge_chunks1024.index                       1.3 MB
  ✓ faiss_bge_chunks256.index                        5.1 MB
  ✓ faiss_bge_chunks512.index                        2.4 MB
  ✓ faiss_minilm_chunks1024.index                    1.3 MB
  ✓ faiss_minilm_chunks256.index                     5.1 MB
  ✓ faiss_minilm_chunks512.index                     2.4 MB

ChromaDB folder:
  ✓ chromadb/ folder exists a

In [11]:
!git clone https://github.com/lakshsharda/GenerativeAI_Assignment

Cloning into 'GenerativeAI_Assignment'...


In [19]:
from google.colab import files
files.download('GENAIASIGNMENT1.ipynb')

FileNotFoundError: Cannot find file: GENAIASIGNMENT1.ipynb

In [18]:
!cp /content/GENAIASIGNMENT1.ipynb assignment1/

cp: cannot stat '/content/GENAIASIGNMENT1.ipynb': No such file or directory


%cd GenerativeAI_Assignment

In [14]:
!mkdir assignment1

In [17]:
!ls /content

drive  GenerativeAI_Assignment	sample_data


In [16]:
!ls

assignment1


In [15]:
!cp /content/GENAIASIGNMENT1.ipynb assignment1/

cp: cannot stat '/content/GENAIASIGNMENT1.ipynb': No such file or directory
